# EXP-2026-002: supplied postoperative scenarios

This notebook calls Python functions for a sampling-only text screen. It is separate from Claude's text-control/caption-feasibility experiment.

Inputs: standalone **preoperative T1ce + preoperative mask + the user's requested scenario**. No actual postoperative image/label is supplied to the VLM or scored as a unique answer. The three scenarios request substantial, minimal and absent enhancing residual, respectively; no RANO class 1 is assigned.

Push the EXP-2026-002 Python/config files to your fork before cloning. MedGemma requires access granted to your Hugging Face account. Models run inside this approved Colab runtime. Keep all run outputs private.


In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "exp-2026-002-scenario-probe"
WORK = "/content/MedSegFactory-scenario-probe"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
RUN_TAG = "scenario_r1"  # A new tag for each attempt; never overwrite runs.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK, ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    active_branch = subprocess.run(["git", "-C", WORK, "branch", "--show-current"], capture_output=True, text=True, check=True).stdout.strip()
    assert active_branch == BRANCH, "WORK belongs to another branch; choose a separate WORK folder"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
link = Path(WORK, "data/RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "SimpleITK==2.5.6", "nibabel>=5.3,<6", "pydicom>=3,<4",
                "scikit-image>=0.25", "matplotlib", "pillow"], check=True)


The usual Hugging Face login or Colab `HF_TOKEN` secret must have access to MedGemma. A gated-model error means access is missing; the runner will retain a private failure record. Use a new run tag after resolving it. Do not paste credentials into a committed notebook.


In [ ]:
from rhuh_synthesis.scenario_probe import run_scenario_probe

DATASET = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
OUTPUT = Path("data/RHUH-GBM/synthesis/EXP-2026-002") / RUN_TAG
assert (DATASET / "stage4/r3/unet_step000500.pt").exists(), "Saved r3 UNet is missing on Drive"
assert not OUTPUT.exists(), "Choose a new RUN_TAG; existing records are immutable"

record = run_scenario_probe(
    "configs/experiments/EXP-2026-002.json",
    dataset_root=DATASET, output_dir=OUTPUT, device="cuda",
)


Outputs: raw VLM JSON/drafts and validation in `captions/`; generated arrays and per-sample records in `arrays/`; prompt-captioned comparisons in `grids/`; paired patient summaries in `record.json`.

The first screen uses seed 11 and at most 336 planned conditions. Identical shuffled prompts are marked inapplicable. Rejected captions stay in the denominator. Passing JSON/keyword checks is not clinical verification. The checkpoint was trained on short templates, so this tests inference-time text sensitivity, not which caption-training strategy is best.


In [ ]:
print({key: record[key] for key in ("status", "patient_count", "slice_count", "attempted", "generated", "caption_rejected", "skipped_inapplicable")})
print("Private results:", OUTPUT)
